# Text Generation with Google GenAI SDK

This notebook demonstrates the new Google GenAI SDK that replaces the older VertexAI library. The GenAI SDK provides a unified interface for both Google AI Studio and Vertex AI services.

## Key Differences from VertexAI Library:
- Unified client for multiple Google AI services
- Simplified authentication using environment variables or direct parameters
- Enhanced configuration options for safety, performance, and behavior control


In [2]:
from google import genai
from google.genai import types
import base64

## Client Configuration

### Parameters Explained:
- `vertexai=True`: Routes requests through Vertex AI (enterprise) instead of Google AI Studio (consumer)
- `project`: Your Google Cloud Project ID for billing and access control
- `location`: Regional endpoint for the API (`global` works for most use cases)


In [3]:
client = genai.Client(
    vertexai=True,
    project="pragmatic-ruler-464913-c3",
    location="global",
)


## Content Creation

### Content Structure:
- `Content`: Represents a message with a role (user/model) and parts
- `Part`: Individual components like text, images, audio, or video
- `from_text()`: Creates a text part
- `from_uri()`: Creates a part from a file URI (Google Cloud Storage)


In [4]:
# Example: Creating multimodal content (text + audio)
text_part = types.Part.from_text(
    text="Please analyze this audio file and summarize the contents as bullet points."
)

audio_part = types.Part.from_uri(
    file_uri="gs://cloud-samples-data/generative-ai/audio/Accessible_writing_tip_Informative_semantic_titles_and_headings.mp3",
    mime_type="audio/mpeg",
)

contents = [
    types.Content(
        role="user",
        parts=[
            text_part,
            audio_part
        ]
    )
]


## Generation Configuration

### Core Parameters:
- `temperature`: Controls randomness (0.0 = deterministic, 1.0 = creative)
- `top_p`: Nucleus sampling threshold (probability cutoff for token selection).
Top-p changes how the model selects tokens for output. Tokens are selected from most probable to least until the sum of their probabilities equals the top-p value. For example, if tokens A, B, and C have a probability of .3, .2, and .1 and the top-p value is .5, then the model will select either A or B as the next token (using temperature). For the least variable results, set top-P to 0.
- `max_output_tokens`: Maximum length of generated response

### Safety Settings:
- `threshold`: defines how sensitive the filter is. 
- There are several levels, like BLOCK_ONLY_HIGH, BLOCK_MEDIUM_AND_ABOVE, and BLOCK_NONE
- `threshold="OFF"`: Disables safety filter for specific harm categories
- **Use with caution**: Your application becomes responsible for content moderation
- Categories: `HATE_SPEECH`, `DANGEROUS_CONTENT`, `SEXUALLY_EXPLICIT`, `HARASSMENT`

### Thinking Configuration:
- `thinking_config`: Controls tool use and function calling behavior
- `thinking_budget`: Limits internal reasoning steps (0 = disable tool use)


In [5]:
generate_content_config = types.GenerateContentConfig(
    temperature=1,
    top_p=0.95,
    max_output_tokens=65535,
    
    # Safety settings - disabling filters for demonstration
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF")
    ],
    
    # Thinking configuration - disabled for simple text generation
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)


## Content Generation Methods

### Streaming vs Non-Streaming:
- `generate_content()`: Returns complete response at once
- `generate_content_stream()`: Returns response in chunks as it's generated
- **Streaming benefits**: Lower latency, real-time feedback, better user experience

### Model Selection:
- `gemini-2.5-flash`: Fast, efficient for most tasks
- `gemini-2.5-flash-lite`: Lightweight version for simple tasks
- `gemini-2.5-pro`: Most capable, slower, higher cost


In [6]:
# Streaming generation example
model = "gemini-2.5-flash-lite"

for chunk in client.models.generate_content_stream(
    model=model,
    contents=contents,
    config=generate_content_config,
):
    print(chunk.text, end="")


Here's a summary of the audio file in bullet points:

*   **Use informative language and semantic code for titles and headings:** This is crucial for content organization and readability, benefiting users with and without accessibility needs.
*   **Headings should be short, high-level summaries:** They should provide a clear expectation of what to find in an article or section, requiring less cognitive effort and being understandable without additional context.
*   **Headings should be action-oriented:** They should tell users what they will learn or what action they should take. Examples include "Get started with this feature" or "Store your data on a server." This doesn't always mean starting with a verb, but the heading should indicate the purpose.
*   **Content structure is as important as content itself:** Headings aid visual navigation, but consider accessibility beyond just visual needs.
*   **Write semantic HTML:** This means using HTML tags that correctly align with the value 

## Non-Streaming Generation

Non-streaming generation waits for the complete response before returning. Useful when you need the full response for processing.


In [7]:
# Non-streaming generation example
simple_content = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text="Recommend a book similar to 'Where the Crawdads Sing'.")]
    )
]

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=simple_content,
    config=generate_content_config,
)

print(response.text)


"Where the Crawdads Sing" by Delia Owens is a very popular book, and it's often praised for its unique blend of nature writing, coming-of-age story, mystery, and evocative setting. Finding something *exactly* like it can be tricky, but here are some recommendations that capture different aspects of what makes "Where the Crawdads Sing" so special:

**For the Evocative Setting and Nature Immersion:**

*   **"The Overstory" by Richard Powers:** While much grander in scope and more philosophical, "The Overstory" is a masterpiece of nature writing. It explores the interconnectedness of humans and trees through multiple interconnected stories. If you loved the deep dive into the natural world of the marsh, you'll appreciate the profound beauty and scientific wonder here.
*   **"Prodigal Summer" by Barbara Kingsolver:** Kingsolver is a master of setting her stories in the natural world, often focusing on rural life and the rhythms of nature. "Prodigal Summer" has a similar feeling of characte

## Key Takeaways

### Migration from VertexAI Library:
- Replace `vertexai.init()` with `genai.Client(vertexai=True)`
- Use `types.Content` and `types.Part` for structured input
- Configure safety and thinking settings explicitly

### Best Practices:
- Use streaming for interactive applications
- Set `thinking_budget=0` if not using function calling
- Be cautious when disabling safety filters
- Choose appropriate model based on speed/capability needs

### Authentication:
- Set environment variables: `GOOGLE_CLOUD_PROJECT`, `GOOGLE_APPLICATION_CREDENTIALS`
- Or pass credentials directly to the client
